# Phase 3c: Export `all-MiniLM-L6-v2` to ONNX for the Android app

**What this notebook produces**
- `minilm.onnx` (~22 MB f32, or ~6 MB int8) — sentence embedding model
- `minilm_vocab.txt` — BERT WordPiece vocabulary
- `minilm_tokenizer_config.json` — uncased flag + special tokens

Drop both files into `/sdcard/Android/data/com.secondbrain.app/files/models/`
via `adb push`, exactly like the GGUF.

**Why this model and not the multilingual one?**
Pure-Kotlin tokenizers for BERT WordPiece are simple (~150 lines).
XLM-RoBERTa SentencePiece Unigram for the multilingual MiniLM is
300-500 lines of Viterbi-decode code, and shipping `huggingface/tokenizers`
Rust JNI adds ~12 MB to the APK. Phase 3c picks the smaller, lower-risk path.
We can swap to multilingual once dogfooding shows English quality is the
actual bottleneck.

**Output dimensions**: 384-float vector, mean-pooled over token outputs,
L2-normalized. The Kotlin runtime applies the same pooling + norm.

**Hardware**: CPU is fine. T4 free tier just makes the export slightly faster.

## CONFIG

In [ ]:
from pathlib import Path
MODEL_ID = 'sentence-transformers/all-MiniLM-L6-v2'
WORK = Path('/content/work_minilm'); WORK.mkdir(exist_ok=True)
ONNX_OUT = WORK / 'minilm.onnx'
VOCAB_OUT = WORK / 'minilm_vocab.txt'
TOKENIZER_CONFIG_OUT = WORK / 'minilm_tokenizer_config.json'
QUANTIZE_INT8 = True   # set False to keep f32; int8 saves ~16 MB at modest accuracy cost
MAX_SEQ_LEN = 256       # matches what we'll use on device
print(WORK, ONNX_OUT, VOCAB_OUT)

## Step 1 — Install dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q transformers==4.44.2 'optimum[exporters]==1.22.0' onnx==1.16.2 onnxruntime==1.18.1 sentence-transformers==3.0.1 numpy==1.26.4

## Step 2 — Export the model to ONNX with optimum

This produces an ONNX graph with two inputs (`input_ids`, `attention_mask`)
and one output (`last_hidden_state`, shape `[batch, seq_len, 384]`).
Mean-pooling + L2-norm is **not** baked into the graph; we do those steps
in Kotlin. That keeps the graph identical to what the model was trained
with and means a future tokenizer swap doesn't require re-export.

In [ ]:
from optimum.onnxruntime import ORTModelForFeatureExtraction
from transformers import AutoTokenizer

model = ORTModelForFeatureExtraction.from_pretrained(MODEL_ID, export=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model.save_pretrained(str(WORK / 'export'))
tokenizer.save_pretrained(str(WORK / 'export'))
import shutil
shutil.copy(WORK / 'export' / 'model.onnx', ONNX_OUT)
print('exported ->', ONNX_OUT)
!ls -lh "{ONNX_OUT}"

## Step 3 — (Optional) Quantize to int8

Brings the file from ~88 MB f32 down to ~23 MB int8 with very small
accuracy regression on sentence-similarity. Skip if you want pristine f32.

In [ ]:
if QUANTIZE_INT8:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    Q_OUT = WORK / 'minilm_int8.onnx'
    quantize_dynamic(
        model_input=str(ONNX_OUT),
        model_output=str(Q_OUT),
        weight_type=QuantType.QUInt8,
    )
    # Replace the f32 export with the quantized one
    Q_OUT.replace(ONNX_OUT)
    print('quantized in place')
!ls -lh "{ONNX_OUT}"

## Step 4 — Save the WordPiece vocab + minimal tokenizer config

The Kotlin tokenizer reads `vocab.txt` (one token per line) and a tiny
JSON file telling it: do_lower_case, unk_token, pad_token, cls_token,
sep_token. This avoids parsing HF's full `tokenizer.json`.

In [ ]:
import json
(WORK / 'export' / 'vocab.txt').replace(VOCAB_OUT)
tcfg = {
    'do_lower_case': bool(getattr(tokenizer, 'do_lower_case', True)),
    'unk_token': tokenizer.unk_token,
    'pad_token': tokenizer.pad_token,
    'cls_token': tokenizer.cls_token,
    'sep_token': tokenizer.sep_token,
    'max_seq_len': MAX_SEQ_LEN,
    'embedding_dim': 384,
    'pooling': 'mean',
    'l2_normalize': True,
}
TOKENIZER_CONFIG_OUT.write_text(json.dumps(tcfg, indent=2))
print(tcfg)
!head -5 "{VOCAB_OUT}"; echo '---'; wc -l "{VOCAB_OUT}"

## Step 5 — Sanity-test the ONNX + tokenizer pair

Embeds two sentences, computes cosine similarity. Should be > 0.5 for
semantically similar sentences and < 0.3 for unrelated.

In [ ]:
import numpy as np
import onnxruntime as ort
sess = ort.InferenceSession(str(ONNX_OUT), providers=['CPUExecutionProvider'])

def embed(text):
    enc = tokenizer(text, padding='max_length', truncation=True, max_length=MAX_SEQ_LEN, return_tensors='np')
    out = sess.run(None, {'input_ids': enc['input_ids'].astype(np.int64),
                          'attention_mask': enc['attention_mask'].astype(np.int64)})
    last = out[0][0]                                 # (seq, 384)
    mask = enc['attention_mask'][0][:, None].astype(np.float32)
    pooled = (last * mask).sum(axis=0) / mask.sum().clip(min=1.0)  # mean pool
    pooled /= np.linalg.norm(pooled) + 1e-12
    return pooled

a = embed('How much did I spend on petrol last month?')
b = embed('Show me transport expenses for last month')
c = embed('Recipe for dosa batter')
print('similar:    ', float(a @ b))
print('unrelated:  ', float(a @ c))

## Step 6 — Download both files

Push them to the phone with:
```
adb push minilm.onnx       /sdcard/Android/data/com.secondbrain.app/files/models/
adb push minilm_vocab.txt  /sdcard/Android/data/com.secondbrain.app/files/models/
adb push minilm_tokenizer_config.json /sdcard/Android/data/com.secondbrain.app/files/models/
```

In [ ]:
try:
    from google.colab import files  # type: ignore
    files.download(str(ONNX_OUT))
    files.download(str(VOCAB_OUT))
    files.download(str(TOKENIZER_CONFIG_OUT))
except Exception as exc:
    print(f'Auto-download not available ({exc}). Files are at:')
    print('  ', ONNX_OUT, '\n  ', VOCAB_OUT, '\n  ', TOKENIZER_CONFIG_OUT)